In [2]:
import pickle
import pprint

# load database.pickle
filename = "/home/yuflin/dev/Gausskit/test_hindrot/database.pickle"
with open(filename, "rb") as f:
    database = pickle.load(f)

# show the data structure of database
pprint.pprint(database)


{'HCFC132b': {'B3LYP': Molecules(symbols='CH2CCl2F2', pbc=False),
              'B3LYP_hindrot': Molecules(symbols='CH2CCl2F2', pbc=False)},
 'HCFC132b-OH_ts': {'MP2_hindrot': Molecules(symbols='CH2CCl2F2OH', pbc=False)},
 'HCFC133a': {'B3LYP': Molecules(symbols='CH2CClF3', pbc=False),
              'B3LYP_hindrot': Molecules(symbols='CH2CClF3', pbc=False)},
 'HCFC133a-OH_ts': {'B3LYP': Molecules(symbols='CH2CClF2OHF', pbc=False)},
 'Mol1': {'B3LYP': Molecules(symbols='CH2CCl2F2', pbc=False)},
 'Radical132b': {'B3LYP': Molecules(symbols='CHCCl2F2', pbc=False)},
 'Radical133a': {'B3LYP': Molecules(symbols='CHCClF3', pbc=False)},
 'TS2': {'B3LYP': Molecules(symbols='CH2CCl2F2OH', pbc=False)}}


In [3]:
# some setting for pretty printing
import numpy as np
import pandas as pd
np.set_printoptions(precision=3)
pd.set_option("display.chop_threshold", 3)
pd.set_option('display.width', 80)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 6)


# show one example in data
print(database["HCFC132b-OH_ts"])
mol = database["HCFC132b-OH_ts"]["MP2_hindrot"] # G4 or MP2,MP2(FC) or B3LYP
# hint:  for  "HCFC132b-OH_ts":
#                 "B3LYP_hindrot":  example with hindered rotor calculation
#                 "B3LYP":          example with anharmonic vibrations
#                 "MP2,MP2(FC)":    example with anharmonic vibrations



# show the basic information (inherited from ase.Atoms)
print(f"Formula: {mol.get_chemical_formula()}")
print(f"Sybmols: {mol.get_chemical_symbols()}")
print("Geometry:")
print(mol.get_positions())


# show the calculation infomation
print(f"Method: {mol.method}")
print(mol.logpath)
# print(f"Basis sets: {mol.basis}") # Bugs
print(f"Electronic energy(hartree): {mol.electronic_energy}")
print(f"ZPE(hartree): {mol.zpe}")
print(f"Is TS calculation? {mol.ts}")
print("Frequencies(cm-1):")
print(mol.frequencies)
print(f"Anharmonic ZPE(hartree): {mol.anharm_zpe}")
print("Anharminic matrix:")
print(pd.DataFrame(mol.anharm_matrix))


# hindered rotor
mol_hindrot = mol.hinderedrotor
if mol_hindrot != []:
    print("Hindered rotor information")
    print("Reduced momentums (amu*Bohr**2)")
    print(mol_hindrot._reduced_moms)
    print("Rotating bond")
    print(mol_hindrot._rotating_bonds)
    print("Rotating atoms")
    print(mol_hindrot._rotating_groups)
    print("In replacement of these vibrations")
    print(mol_hindrot._corrected_vibs)
    print(mol_hindrot._symmetry_numbers)

{'MP2_hindrot': Molecules(symbols='CH2CCl2F2OH', pbc=False)}
Formula: C2H3Cl2F2O
Sybmols: ['C', 'H', 'H', 'C', 'Cl', 'Cl', 'F', 'F', 'O', 'H']
Geometry:
[[ 0.     0.     0.   ]
 [ 0.     0.     1.211]
 [ 1.041  0.    -0.315]
 [-0.727  1.254 -0.43 ]
 [ 0.19   2.677  0.083]
 [-0.832 -1.44  -0.536]
 [-0.889  1.295 -1.751]
 [-1.944  1.297  0.131]
 [-0.348  0.218  2.414]
 [-1.278  0.44   2.26 ]]
Method: MP2
/home/yuflin/dev/Gausskit/test_hindrot/hindrot/HCFC132b-OH_ts.log
Electronic energy(hartree): -1271.4393205249
ZPE(hartree): 0.048777
Is TS calculation? True
Frequencies(cm-1):
[-2204.6    74.3    92.9   142.    177.1   187.8   286.2   310.9   408.5
   446.6   566.    733.9   772.2   821.4   873.8   987.6  1063.4  1083.4
  1232.6  1267.5  1327.6  1527.6  3212.1  3814.3]
anharmonic zero point energy is not set
Anharmonic ZPE(hartree): None
Anharminic matrix:
anharmonic X matrix is not set
Empty DataFrame
Columns: []
Index: []
Hindered rotor information
Reduced momentums (amu*Bohr**2)
[17.

In [5]:
# hindered rotor
filename = "/home/yuflin/dev/Gausskit/test_hindrot/thermo_PES_2/Mol1.coords"
# load file
with open(filename, "r") as f:
    lines = f.readlines()

# internal rotor coords information
internal_rotor_coords = ['']
for i_rotor in range(len(mol_hindrot._rotating_bonds)):
    internal_rotor_coords.append(', '.join([str(j+1) for j in mol_hindrot._rotating_bonds[i_rotor]]))
    internal_rotor_coords.append(str(len(mol_hindrot._rotating_groups[i_rotor])))
    internal_rotor_coords.append(', '.join([str(j+1) for j in mol_hindrot._rotating_groups[i_rotor]]))
    internal_rotor_coords.append('')

internal_rotor_coords = [j + '\n' for j in internal_rotor_coords]

# insert rotor coords information to the file
n_line = lines.index(' 0 , 0\n')
new_lines = lines[:n_line] + internal_rotor_coords + lines[n_line:]


output = "Mol1.coords"
with open(output, "w") as f:
    f.writelines(new_lines)


['\n', '1, 2\n', '2\n', '9, 10\n', '\n', '1, 4\n', '5\n', '2, 3, 6, 9, 10\n', '\n', '2, 9\n', '1\n', '10\n', '\n']
11
new_lines
[' Mol1\n', ' ANGS\n', '  8\n', '  C    1   -0.616491   -0.868795    0.000060\n', '  H    2   -0.494765   -1.477366    0.894213\n', '  H    3   -0.494777   -1.477432   -0.894049\n', '  C    4    0.418975    0.246605    0.000011\n', ' Cl    5    2.067807   -0.511527    0.000042\n', ' Cl    6   -2.277221   -0.191314    0.000046\n', '  F    7    0.315428    1.019012   -1.079780\n', '  F    8    0.315431    1.019108    1.079732\n', '\n', '1, 2\n', '2\n', '9, 10\n', '\n', '1, 4\n', '5\n', '2, 3, 6, 9, 10\n', '\n', '2, 9\n', '1\n', '10\n', '\n', ' 0 , 0\n', '\n', '\n', ' Please check internal rotors\n', '\n']
lines
[' Mol1\n', ' ANGS\n', '  8\n', '  C    1   -0.616491   -0.868795    0.000060\n', '  H    2   -0.494765   -1.477366    0.894213\n', '  H    3   -0.494777   -1.477432   -0.894049\n', '  C    4    0.418975    0.246605    0.000011\n', ' Cl    5    2.067807  